In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import argparse
import ast


from AIedes.data_loader.classificator_data_loader import normalize_temperature_df, normalize_precipitation_df, get_data_loaders

from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from AIedes.train.classificator_train    import run_model
from AIedes.evaluation.classificator_evaluate import evaluate_model_nuts3, evaluate_model



In [4]:
batch_size = 256
learning_rate = 1e-3
num_epochs = 100
test_ratio = 0.15
val_ratio = 0.15
random_seed = 42


In [6]:
parent_dir = "./data/"
year = "2020"
df = pd.read_csv(parent_dir + f'albopictus_presence_absence_ecdc_world_clim_all_bio.zip')
df_suitable = df

# Filter data based on presence/absence and suitability
df_Presence_Absence = df_suitable[(df_suitable["Presence_Absence"] == 1) | (df_suitable["Presence_Absence"] == 0)].copy()

# Set presence/absence to 1 if suitable is 1, otherwise set to 0
df_Presence_Absence["Presence_Absence"] = ((df_Presence_Absence['Presence_Absence'] == 1) & (df_Presence_Absence['Suitable'] == 1)).astype(int)

# Convert to GeoDataFrame and filter absence points near presence
df_gdf = gpd.GeoDataFrame(
    df_Presence_Absence,
    geometry=gpd.points_from_xy(df_Presence_Absence['Longitude'], df_Presence_Absence['Latitude'])
)

# Assuming `gdf` is your GeoDataFrame with a 'record' column that indicates 'presence' or 'absence'
df_gdf.set_crs(epsg=4326, inplace=True)

df_gdf = df_gdf.to_crs(epsg=3857)

# Step 1: Split into presence and absence records
presence_gdf = df_gdf[df_gdf['Presence_Absence'] == 1]
absence_gdf = df_gdf[df_gdf['Presence_Absence'] == 0]

# Step 2: Buffer presence points by 1 km (distance in meters; assumes CRS in meters)
presence_buffered = presence_gdf.copy()
presence_buffered['geometry'] = presence_buffered.geometry.buffer(1000)  # 1 km buffer
print("Buffered presence points")
# Step 3: Spatial join to find absence points within 1 km of any presence point
near_presence = gpd.sjoin(absence_gdf, presence_buffered, how='inner', predicate='within')
print("Absence points near presence points")
# Step 4: Remove these absence points from the original GeoDataFrame
filtered_gdf = df_gdf[~df_gdf.index.isin(near_presence.index)]


df_Presence_Absence = filtered_gdf

# Feature selection

#features = ['annual_temp', 'annual_prec', 'max_temp_warmest_month', 'prec_warmest_quarter', 'TAVG_01', 'TAVG_02', 'TAVG_03', 'TAVG_04', 'TAVG_05', 'TAVG_06', 'TAVG_07', 'TAVG_08', 'TAVG_09', 'TAVG_10', 'TAVG_11', 'TAVG_12']

features = ['annual_temp', 'annual_prec', 'max_temp_warmest_month', 'prec_warmest_quarter',
        'TAVG_01', 'TAVG_02', 'TAVG_03', 'TAVG_04', 'TAVG_05', 'TAVG_06', 'TAVG_07', 
        'TAVG_08', 'TAVG_09', 'TAVG_10', 'TAVG_11', 'TAVG_12',
        'BIO_2', 'BIO_3', 'BIO_4','BIO_6', 'BIO_7',
        'BIO_8', 'BIO_9', 'BIO_10', 'BIO_11', 'BIO_13', 'BIO_14',
        'BIO_15', 'BIO_16', 'BIO_17']


data = df_Presence_Absence[features + ["Presence_Absence"]].to_numpy()
data_four = df_Presence_Absence[features[0:4] + ["Presence_Absence"]].to_numpy()
train_loader, test_loader, _, input_size = get_data_loaders(
    data=data, batch_size = batch_size, test_ratio = test_ratio, 
    random_seed = random_seed, shuffle = True
)


Buffered presence points
Absence points near presence points


/home/biazzin/git/AIedes/src/AIedes/data_loader/classificator_data_loader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.y = torch.tensor(y, dtype=torch.float)  # Labels for classification
